# EBSD Regular-Grid Workflows

This notebook walks through the current EBSD foundation:

- regular-grid neighborhood construction
- KAM
- grain segmentation
- GROD
- grain-boundary and grain-graph summaries


In [1]:
from pathlib import Path
import tempfile

import numpy as np

from pytex import (
    AcquisitionGeometry,
    AtomicSite,
    BenchmarkManifest,
    build_crystal_scene,
    CalibrationRecord,
    CrystalCellOverlay,
    CrystalDirection,
    CrystalDirectionOverlay,
    CrystalMap,
    CrystalPlane,
    CrystalPlaneOverlay,
    DirectionAnnotationStyle,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    get_phase_fixture,
    list_phase_fixtures,
    list_style_themes,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    PowderPattern,
    PowderReflection,
    ReferenceFrame,
    read_validation_manifest,
    read_workflow_result_manifest,
    resolve_style,
    RadiationSpec,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    UnitCell,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    PlaneAnnotationStyle,
    generate_saed_pattern,
    generate_xrd_pattern,
    normalize_ebsd,
    plot_odf,
    plot_crystal_structure_3d,
    plot_inverse_pole_figure,
    plot_ipf_map,
    plot_orientations,
    plot_kam_map,
    plot_pole_figure,
    plot_saed_pattern,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
    plot_xrd_pattern,
)


def make_crystal_frame():
    return ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )


def make_context():
    crystal = make_crystal_frame()
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    phase = get_phase_fixture("ni_fcc").load_phase(crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


def describe_phase_fixture(fixture_id):
    record = get_phase_fixture(fixture_id)
    return {
        "fixture_id": record.fixture_id,
        "display_name": record.display_name,
        "artifact_path": str(record.artifact_path),
        "metadata_path": str(record.metadata_path),
        "intended_uses": tuple(record.metadata["intended_uses"]),
    }


def load_zr_hcp_phase():
    return get_phase_fixture("zr_hcp").load_phase(crystal_frame=make_crystal_frame())


def load_diamond_phase():
    return get_phase_fixture("diamond").load_phase(crystal_frame=make_crystal_frame())


def publication_crystal_style():
    return {
        "crystal": {
            "atom_radius_scale": 0.5,
            "atom_edgewidth": 0.0,
            "atom_surface_resolution": 34,
            "bond_surface_resolution": 28,
            "bond_alpha": 0.72,
            "bond_color": "#7c8ea3",
            "atom_specular_strength": 0.42,
            "light_specular": 0.4,
        }
    }


In [2]:
crystal, specimen, map_frame, detector, lab, phase = make_context()
orientations = OrientationSet.from_euler_angles(
    np.array(
        [
            [0.0, 0.0, 0.0],
            [2.0, 0.0, 0.0],
            [25.0, 0.0, 0.0],
            [27.0, 0.0, 0.0],
        ]
    ),
    crystal_frame=crystal,
    specimen_frame=specimen,
    symmetry=phase.symmetry,
    phase=phase,
)

crystal_map = CrystalMap(
    coordinates=np.array(
        [
            [0.0, 0.0],
            [1.0, 0.0],
            [0.0, 1.0],
            [1.0, 1.0],
        ]
    ),
    orientations=orientations,
    map_frame=map_frame,
    grid_shape=(2, 2),
    step_sizes=(1.0, 1.0),
    # Orientations live in the specimen frame while the scan grid lives in the
    # map frame. PyTex refuses to guess how those frames relate, so an explicit
    # specimen -> map transform is required before the map can be exported.
    acquisition_geometry=AcquisitionGeometry(
        specimen_frame=specimen,
        modality="ebsd",
        map_frame=map_frame,
        specimen_to_map=FrameTransform(
            source=specimen,
            target=map_frame,
            rotation_matrix=np.eye(3),
        ),
    ),
)

print("Neighbor pairs")
print(crystal_map.neighbor_pairs())
print("KAM")
print(crystal_map.kernel_average_misorientation_deg(symmetry_aware=False))


Neighbor pairs
[[0 1]
 [2 3]
 [0 2]
 [1 3]]
KAM
[[13.5 13.5]
 [13.5 13.5]]


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:720: UserWarning: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn(msg)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/pymatgen/io/cif.py:1224: UserWarning: Issues encountered while parsing CIF: No _symmetry_equiv_pos_as_xyz type key found. Spacegroup from _symmetry_space_group_name_H-M used.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))


In [3]:
segmentation = crystal_map.segment_grains(
    max_misorientation_deg=5.0,
    symmetry_aware=False,
)
print("Labels")
print(segmentation.label_grid)
print("GROD")
print(segmentation.grod_map_deg())

boundary_network = segmentation.boundary_network(min_misorientation_deg=5.0)
print("Boundary count:", boundary_network.count)
print("Graph adjacency")
print(boundary_network.grain_graph().adjacency_matrix)


Labels
[[0 0]
 [1 1]]
GROD
[[0. 2.]
 [0. 2.]]
Boundary count: 2
Graph adjacency
[[0 1]
 [1 0]]


In [4]:
experiment = crystal_map.to_experiment_manifest(source_system="pytex")
print(experiment.to_dict()["modality"])
print(experiment.to_dict()["metadata"])


ebsd
{'grid_shape': '2x2', 'step_sizes': '1,1'}
